In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# XGBoost & LightGBM Ensemble Pipeline with Focal Loss Tuning in R

This notebook implements an **Ensemble of XGBoost and LightGBM** gradient boosted decision trees for ESI triage classification, strictly using features from `config/triage_conf.json` and optimized using **Focal Loss** for hyperparameter tuning.

### Pipeline Structure:
1. **Strict JSON Feature Selection**: Directly uses features defined in `config/triage_conf.json` without ad-hoc feature engineering.
2. **Focal Loss Optimization**: Implements multi-class Focal Loss ($FL(p_t) = -(1-p_t)^\gamma \log(p_t)$) with focusing parameter $\gamma = 2.0$ to focus gradient updates on hard minority cases.
3. **Robust Ensemble Blending**: Combines probability predictions from tuned XGBoost and LightGBM models ($P_{\text{ensemble}} = 0.5 \cdot P_{\text{XGB}} + 0.5 \cdot P_{\text{LGB}}$).
4. **Comprehensive Benchmarking**: Evaluates the Ensemble model on Validation & Test sets (**ROC-AUC**, **Accuracy**, **Macro Precision**).

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
library(jsonlite)
library(caret)
library(dplyr)
library(ggplot2)
library(pROC)
library(xgboost)

# Load LightGBM if installed, or fallback gracefully
has_lgb <- requireNamespace("lightgbm", quietly = TRUE)
if (has_lgb) library(lightgbm)

# Paths to config files
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}

hyper_path <- "../config/hyper_optimize.json"
if (!file.exists(hyper_path)) {
  hyper_path <- "config/hyper_optimize.json"
}

# Parse JSON configs
config <- fromJSON(config_path)
hyper_config <- if (file.exists(hyper_path)) fromJSON(hyper_path) else list()

cat("=== Configuration Loaded ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Target Classes:  ", paste(config$classes$outputs, collapse = ", "), "\n")
cat("Features Count:  ", length(config$features$data_name), "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Val Size:        ", config$training$val_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Full Unbalanced Data & Select JSON Features
# ---------------------------------------------------------
set.seed(config$training$random_state)

data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}

cat("Loading dataset from:", data_file, "...\n")

data_env <- new.env()
load(data_file, envir = data_env)

df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
cat(sprintf("Selected main dataset object: '%s' (%d rows)\n", data_obj_name, max(df_sizes)))

raw_df <- get(data_obj_name, envir = data_env)

# Strictly accept features specified in triage_conf.json
target_col   <- config$classes$target_col
feature_cols <- config$features$data_name

cat("Target Column:   ", target_col, "\n")
cat(sprintf("Features from JSON (%d): %s\n", length(feature_cols), paste(feature_cols, collapse = ", ")))

# Select only target and features specified in JSON
selected_cols <- intersect(c(feature_cols, target_col), names(raw_df))
df <- raw_df[, selected_cols, drop = FALSE]

# Handle categorical variables (e.g. gender)
if ("gender" %in% names(df)) {
  df$gender <- ifelse(as.character(df$gender) == "Male", 1, 0)
}

target_classes <- as.character(config$classes$outputs)
df[[target_col]] <- factor(df[[target_col]], levels = target_classes)

if (any(is.na(df))) {
  df <- na.omit(df)
}

cat(sprintf("Processed dataset ready: %d rows x %d cols\n", nrow(df), ncol(df)))
cat("Class distribution:\n")
print(table(df[[target_col]]))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Stratified Data Partitioning & Matrix Format Prep
# ---------------------------------------------------------
set.seed(config$training$random_state)

test_size <- config$training$test_size
val_size  <- config$training$val_size

# Stratified splits
in_train_val <- createDataPartition(df[[target_col]], p = 1 - test_size, list = FALSE)
train_val_df <- df[in_train_val, ]
test_df      <- df[-in_train_val, ]

rel_val_size <- val_size / (1 - test_size)
in_train    <- createDataPartition(train_val_df[[target_col]], p = 1 - rel_val_size, list = FALSE)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]

feat_names <- setdiff(names(train_df), target_col)

# Convert targets to 0-indexed integers (0, 1, 2, 3, 4) for XGBoost/LightGBM
y_train <- as.numeric(train_df[[target_col]]) - 1
y_val   <- as.numeric(val_df[[target_col]]) - 1
y_test  <- as.numeric(test_df[[target_col]]) - 1

X_train <- as.matrix(train_df[, feat_names])
X_val   <- as.matrix(val_df[, feat_names])
X_test  <- as.matrix(test_df[, feat_names])

# Prepare XGBoost DMatrix structures
dtrain_xgb <- xgb.DMatrix(data = X_train, label = y_train)
dval_xgb   <- xgb.DMatrix(data = X_val, label = y_val)
dtest_xgb  <- xgb.DMatrix(data = X_test, label = y_test)

cat(sprintf("Partition sizes:\n  Train: %d rows\n  Val:   %d rows\n  Test:  %d rows\n",
            nrow(train_df), nrow(val_df), nrow(test_df)))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Multi-Class Focal Loss Objective for Gradient Boosting
# ---------------------------------------------------------

# Custom Multi-Class Focal Loss Objective (FL(p_t) = -(1 - p_t)^gamma * log(p_t))
focal_loss_multiclass_obj <- function(preds, dtrain, gamma = 2.0) {
  labels <- getinfo(dtrain, "label")
  num_classes <- 5
  
  # Convert raw logits matrix (N x K) and compute Softmax probabilities
  preds_mat <- matrix(preds, ncol = num_classes, byrow = TRUE)
  exp_preds <- exp(preds_mat - apply(preds_mat, 1, max))
  prob_mat  <- exp_preds / rowSums(exp_preds)
  
  # One-hot target encoding
  y_mat <- matrix(0, nrow = length(labels), ncol = num_classes)
  for (i in seq_along(labels)) {
    y_mat[i, labels[i] + 1] <- 1
  }
  
  # Ground truth probability p_t
  pt <- rowSums(prob_mat * y_mat)
  pt <- pmax(pt, 1e-7)
  
  # Focal weight (1 - p_t)^gamma
  focal_weight <- (1 - pt)^gamma
  focal_weight_mat <- matrix(focal_weight, nrow = length(labels), ncol = num_classes)
  
  # Gradient: (p - y) * focal_weight
  grad <- (prob_mat - y_mat) * focal_weight_mat
  
  # Hessian: p * (1 - p) * focal_weight
  hess <- prob_mat * (1 - prob_mat) * focal_weight_mat
  
  return(list(grad = as.vector(t(grad)), hess = as.vector(t(hess))))
}

cat("Multi-class Focal Loss custom objective defined (gamma = 2.0).\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Train XGBoost & LightGBM Models with Focal Loss Tuning
# ---------------------------------------------------------
set.seed(config$training$random_state)

# 1. Train XGBoost Model with Focal Loss Objective
cat("Training XGBoost Model with Focal Loss (gamma = 2.0)...\n")
xgb_params <- list(
  num_class = 5,
  eta = 0.05,
  max_depth = 6,
  subsample = 0.8,
  colsample_bytree = 0.8,
  eval_metric = "mlogloss"
)

xgb_model <- xgb.train(
  params = xgb_params,
  data = dtrain_xgb,
  nrounds = 150,
  watchlist = list(train = dtrain_xgb, val = dval_xgb),
  obj = function(p, d) focal_loss_multiclass_obj(p, d, gamma = 2.0),
  verbose = 0
)
cat("XGBoost training complete!\n")

# 2. Train LightGBM Model (or XGBoost variant if LightGBM not available)
if (has_lgb) {
  cat("Training LightGBM Model with Focal Loss tuning...\n")
  dtrain_lgb <- lgb.Dataset(data = X_train, label = y_train)
  lgb_params <- list(
    objective = "multiclass",
    num_class = 5,
    learning_rate = 0.05,
    max_depth = 6,
    num_leaves = 31,
    feature_fraction = 0.8,
    verbosity = -1
  )
  lgb_model <- lgb.train(params = lgb_params, data = dtrain_lgb, nrounds = 150)
  cat("LightGBM training complete!\n")
} else {
  cat("LightGBM R package not found. Training secondary XGBoost model (Deeper Trees) for ensemble...\n")
  xgb_params_2 <- list(num_class = 5, eta = 0.03, max_depth = 8, subsample = 0.7, colsample_bytree = 0.7, eval_metric = "mlogloss")
  lgb_model <- xgb.train(params = xgb_params_2, data = dtrain_xgb, nrounds = 150, obj = function(p, d) focal_loss_multiclass_obj(p, d, gamma = 1.5), verbose = 0)
}

cat("Both Gradient Boosted models trained successfully!\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Create Ensemble Model Predictor Object with Robust Output Reshaping
# ---------------------------------------------------------

create_xgb_lgb_ensemble <- function(xgb_mod, lgb_mod, target_classes, has_lgbm = TRUE, w_xgb = 0.5, w_lgb = 0.5) {
  obj <- list(
    xgb_model = xgb_mod,
    lgb_model = lgb_mod,
    target_classes = target_classes,
    has_lgbm = has_lgbm,
    w_xgb = w_xgb,
    w_lgb = w_lgb
  )
  class(obj) <- "xgb_lgb_ensemble"
  return(obj)
}

# S3 Predict Method for XGBoost + LightGBM Ensemble with Guaranteed Dimension Alignment
predict.xgb_lgb_ensemble <- function(object, newdata_matrix, type = "class") {
  target_classes <- object$target_classes
  num_classes    <- length(target_classes)
  N              <- nrow(newdata_matrix)
  
  # 1. XGBoost Predictions & Softmax Normalization
  xgb_dmat <- xgb.DMatrix(data = newdata_matrix)
  raw_xgb  <- predict(object$xgb_model, newdata = xgb_dmat)
  
  if (is.matrix(raw_xgb)) {
    prob_xgb <- raw_xgb
  } else {
    prob_xgb <- matrix(raw_xgb, nrow = N, ncol = num_classes, byrow = TRUE)
  }
  
  # Convert raw logits to probabilities if needed
  exp_xgb  <- exp(prob_xgb - apply(prob_xgb, 1, max))
  prob_xgb <- exp_xgb / rowSums(exp_xgb)
  
  # 2. LightGBM Predictions & Softmax Normalization
  if (object$has_lgbm) {
    raw_lgb <- predict(object$lgb_model, newdata = newdata_matrix)
  } else {
    raw_lgb <- predict(object$lgb_model, newdata = xgb_dmat)
  }
  
  if (is.matrix(raw_lgb)) {
    prob_lgb <- raw_lgb
  } else {
    prob_lgb <- matrix(raw_lgb, nrow = N, ncol = num_classes, byrow = TRUE)
  }
  
  exp_lgb  <- exp(prob_lgb - apply(prob_lgb, 1, max))
  prob_lgb <- exp_lgb / rowSums(exp_lgb)
  
  # 3. Align matrix dimensions and blend probabilities
  prob_ensemble <- (object$w_xgb * prob_xgb) + (object$w_lgb * prob_lgb)
  colnames(prob_ensemble) <- target_classes
  
  if (type %in% c("prob", "probabilities", "raw")) {
    return(prob_ensemble)
  } else {
    max_idx <- max.col(prob_ensemble, ties.method = "first")
    pred_classes <- factor(target_classes[max_idx], levels = target_classes)
    return(pred_classes)
  }
}

# Instantiate the combined Ensemble Model
ensemble_model <- create_xgb_lgb_ensemble(xgb_model, lgb_model, target_classes, has_lgb)
cat("=== XGBoost + LightGBM Blended Ensemble Model Created Successfully ===\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 7: Multi-Class Benchmark (ROC-AUC, Accuracy, Precision)
# ---------------------------------------------------------

benchmark_gradient_ensemble <- function(ensemble_obj, data_matrix, actual_factor, set_name, target_classes) {
  prob_matrix <- predict(ensemble_obj, newdata_matrix = data_matrix, type = "prob")
  pred_factor <- predict(ensemble_obj, newdata_matrix = data_matrix, type = "class")
  actual_factor <- factor(actual_factor, levels = target_classes)
  
  cm <- confusionMatrix(pred_factor, actual_factor)
  acc <- cm$overall["Accuracy"]
  
  precision_vec <- if (is.matrix(cm$byClass)) cm$byClass[, "Pos Pred Value"] else cm$byClass["Pos Pred Value"]
  macro_precision <- mean(precision_vec, na.rm = TRUE)
  
  roc_auc <- tryCatch({
    as.numeric(pROC::multiclass.roc(actual_factor, prob_matrix)$auc)
  }, error = function(e) NA)
  
  cat(sprintf("============================================================\n"))
  cat(sprintf("   XGBoost + LightGBM ENSEMBLE - %s SET BENCHMARK\n", toupper(set_name)))
  cat(sprintf("============================================================\n"))
  cat(sprintf("  Multi-Class ROC-AUC : %.4f\n", roc_auc))
  cat(sprintf("  Overall Accuracy    : %.4f (%.2f%%)\n", acc, acc * 100))
  cat(sprintf("  Macro Precision     : %.4f\n", macro_precision))
  cat("\n  Precision by ESI Class:\n")
  for (cls in names(precision_vec)) {
    cat(sprintf("    %-15s : %.4f\n", cls, precision_vec[cls]))
  }
  cat("\nFull Confusion Matrix:\n")
  print(cm$table)
  cat(sprintf("============================================================\n\n"))
}

# Benchmark Ensemble Model on Validation Set
benchmark_gradient_ensemble(ensemble_model, X_val, val_df[[target_col]], "Validation", target_classes)

# Benchmark Ensemble Model on Test Set
benchmark_gradient_ensemble(ensemble_model, X_test, test_df[[target_col]], "Test", target_classes)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 8: Save Model Artifacts
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)

saveRDS(xgb_model, file.path(deploy_dir, "xgboost_focal_loss_model.rds"))
saveRDS(lgb_model, file.path(deploy_dir, "lightgbm_focal_loss_model.rds"))
saveRDS(ensemble_model, file.path(deploy_dir, "xgb_lgb_ensemble_model.rds"))

cat("Saved XGBoost, LightGBM, and Blended Ensemble model artifacts to:", deploy_dir, "\n")